# Practical 3 — Diamond Dataset Visualization and Linear Regression

## Objective
Plot price by cut, create a frequency table for cut, show carat and depth distributions, and build/evaluate a linear regression model.

**How to use:** Put the required CSV file(s) in the `data/` folder, update the filename/path if needed, and run cells from top to bottom. Where a dataset is unavailable, the notebook reports that clearly rather than inventing results.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
csv_path = DATA_DIR / "diamond.csv"
if csv_path.exists():
    diamonds = pd.read_csv(csv_path)
else:
    try:
        import seaborn as sns
        diamonds = sns.load_dataset("diamonds")
        print("Loaded seaborn diamonds fallback; use the supplied diamond.csv for exact lab outputs.")
    except Exception as exc:
        raise FileNotFoundError("Place diamond.csv in the data/ folder.") from exc
display(diamonds.head())
print(diamonds.dtypes)

In [ ]:
required = {"price", "cut", "carat", "depth"}
missing = required - set(diamonds.columns)
if missing:
    raise KeyError(f"Dataset is missing required columns: {missing}")
diamonds.boxplot(column="price", by="cut", figsize=(9, 5), grid=False)
plt.title("Price by cut")
plt.suptitle("")
plt.xlabel("Cut")
plt.ylabel("Price")
plt.tight_layout()
plt.show()
median_prices = diamonds.groupby("cut", observed=False)["price"].median().sort_values(ascending=False)
display(median_prices.to_frame("median_price"))
print("Cut category/categories with highest median price:", median_prices[median_prices == median_prices.max()].index.tolist())

In [ ]:
cut_frequency = diamonds["cut"].value_counts(dropna=False).rename_axis("cut").to_frame("frequency")
display(cut_frequency)
print("Ideal frequency:", int(diamonds["cut"].astype(str).eq("Ideal").sum()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(diamonds["carat"].dropna(), bins=30, edgecolor="black")
axes[0].set_title("Carat weight distribution")
axes[0].set_xlabel("Carat")
axes[0].set_ylabel("Frequency")
axes[1].hist(diamonds["depth"].dropna(), bins=30, edgecolor="black")
axes[1].set_title("Diamond depth distribution")
axes[1].set_xlabel("Depth")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

## Linear regression model and evaluation

In [ ]:
# Predict price from carat using a reproducible train/test split.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model_data = diamonds[["carat", "price"]].apply(pd.to_numeric, errors="coerce").dropna()
X = model_data[["carat"]]
y = model_data["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
print("R² score (regression goodness of fit):", round(r2_score(y_test, pred), 4))
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("RMSE:", round(mean_squared_error(y_test, pred) ** 0.5, 2))
print("Coefficient:", model.coef_[0], "| Intercept:", model.intercept_)
print("Note: R² is used here; regression is not normally evaluated with classification accuracy.")

## Conclusion

Report the highest median-price cut, Ideal frequency, distribution shapes, and regression metrics. The model predicts price from carat only, so its score is not a universal measure of diamond-price prediction quality.